## PREPROCESSING

In [16]:
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

def load_npy_files(folder):
    """Load all .npy files in a folder into a dict."""
    data = {}
    for f in sorted(folder.glob("*.npy")):   
        # if "had" in f.name:
        #     continue
        data[f.name] = np.load(f)
    return data

# original prep data downloaded from warehouse: 
can_had_v2 = Path('C:/Users/ceidigh/Documents/Preprocess/can_had')
egfp_v2 = Path('C:/Users/ceidigh/Documents/Preprocess/egfp')
mrfp_v2 = Path('C:/Users/ceidigh/Documents/Preprocess/mrfp')

# prep data generated using scripts updated for SPYRiT 3.1
can_had_v3 = Path('data/2023_03_07_mRFP_DsRed_can_vs_had/Preprocess')
egfp_v3 = Path('data/2023_03_13_2023_03_14_eGFP_DsRed_3D/Preprocess')
mrfp_v3 = Path('data/2023_02_28_mRFP_DsRed_3D/Preprocess')

In [3]:
# load data
can_had_data_v2 = load_npy_files(can_had_v2)
egfp_data_v2 = load_npy_files(egfp_v2)
mrfp_data_v2 = load_npy_files(mrfp_v2)


can_had_data_v3 = load_npy_files(can_had_v3)
egfp_data_v3 = load_npy_files(egfp_v3)
mrfp_data_v3 = load_npy_files(mrfp_v3)

print('Can_Had:\n\tFound', len(can_had_data_v2), 'v2 prep files\n\tFound', len(can_had_data_v3), 'v3 prep files')
print('EGFP:\n\tFound', len(egfp_data_v2), 'v2 prep files\n\tFound', len(egfp_data_v3), 'v3 prep files')
print('mRFP:\n\tFound', len(mrfp_data_v2), 'v2 prep files\n\tFound', len(mrfp_data_v3), 'v3 prep files')

Can_Had:
	Found 4 v2 prep files
	Found 4 v3 prep files
EGFP:
	Found 52 v2 prep files
	Found 52 v3 prep files
mRFP:
	Found 61 v2 prep files
	Found 61 v3 prep files


In [7]:
# Compare
def compare(data_v3, data_v2):
    common_files = sorted(set(data_v3) & set(data_v2))
    diff = {}

    for fname in common_files:
        arr_v3 = data_v3[fname]
        arr_v2 = data_v2[fname]

        diff[fname] = arr_v3 - arr_v2

    return diff



In [ ]:

diff_can_had = compare(can_had_data_v3, can_had_data_v2)
diff_egfp = compare(egfp_data_v3, egfp_data_v2)
diff_mrfp = compare(mrfp_data_v3, mrfp_data_v2)

print('Difference between prep data files (v3 - v2)')
print('\tCan_Had: ', sum(arr.sum() for arr in diff_can_had.values()))
print('\tEGFP: ', sum(arr.sum() for arr in diff_egfp.values()))
print('\tmRFP: ', sum(arr.sum() for arr in diff_mrfp.values()))

Difference between prep data files (v3 - v2)
	Can_Had:  0.0
	EGFP:  0.0
	mRFP:  0.0


## RECONSTRUCTION

### FIGURE 3

The pilot warehouse does not contain the corresponding data - I will re run the reconstructions to generate data that matches (*Fellgett Reconstruction*)

In [17]:
v2 = Path('C:/Users/ceidigh/Documents/hypercube')
v3 = Path('data/2023_03_07_mRFP_DsRed_can_vs_had/Reconstruction/hypercube_TEST')

v2_hypercubes = load_npy_files(v2)
v3_hypercubes = load_npy_files(v3)

print('Found', len(v2_hypercubes), 'v2 hypercubes\nFound', len(v3_hypercubes), 'v3 hypercubes')

Found 16 v2 hypercubes
Found 16 v3 hypercubes


In [28]:
common_files = sorted(set(v3_hypercubes) & set(v2_hypercubes))
mse_set = {}

for fname in common_files:
    arr_v3 = v3_hypercubes[fname]
    arr_v2 = v2_hypercubes[fname]

    mse = np.mean((arr_v3 - arr_v2) ** 2)

    mse_set[fname] = mse
sum(mse_set.values()) / len(mse_set)

np.float32(2.0401025e-09)

In [18]:
diff = compare(v3_hypercubes, v2_hypercubes)
print('Difference between felgett hypercubes (v3 - v2)')
print(sum(arr.sum() for arr in diff.values()))

Difference between felgett hypercubes (v3 - v2)
28.162125


In [19]:
total = 0
count = 0

for k in diff:
    v2 = v2_hypercubes[k]
    d  = np.abs(diff[k])

    total += np.sum(d / (np.abs(v2) + 1e-8))
    count += d.size

mape = 100 * total / count
print(f"Mean % difference: {mape:.2f}%")

Mean % difference: 0.00%
